# Academic Journals - SPECTER2 Embedding-Analyse

Wandelt jedes Paper (Titel + Abstract) in einen Zahlen-Vektor um und misst, wie thematisch aehnlich sich Paper sind. Ergebnis: **topic_match**. Abschnitte der Reihe nach ausfuehren.

## 1 - Was macht SPECTER2?

SPECTER2 verwandelt Titel + Abstract eines Papers in einen 768-Zahlen-Vektor. Aehnliche Themen ergeben aehnliche Vektoren. Damit koennen wir thematische Naehe zwischen Papern messen.

## 2 - Welche Autoren?

Wir rechnen ueber alle Autoren mit mindestens 3 Papern. Autoren mit nur einem Paper bringen fuer die Journal-Treue nichts.

## 3 - Datenbank in Colab

Die DuckDB liegt in Google Drive. Wir binden Drive ein und kopieren die Datei in den lokalen Colab-Speicher (schneller). GPU aktivieren: Runtime > Change runtime type > T4 GPU.

In [ ]:
# Google Drive verbinden
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Datenbank aus Drive lokal kopieren (Pfad ggf. anpassen)
import shutil, os

DRIVE_DB = "/content/drive/MyDrive/academic-journals/academic_journals.duckdb"
LOCAL_DB = "/content/academic_journals.duckdb"

assert os.path.exists(DRIVE_DB), f"DB nicht gefunden: {DRIVE_DB}  (Pfad anpassen!)"
shutil.copy(DRIVE_DB, LOCAL_DB)
print("DB kopiert nach", LOCAL_DB, f"({os.path.getsize(LOCAL_DB)/1e6:.0f} MB)")

## 4 - Setup

Pakete installieren und pruefen, ob eine GPU aktiv ist.

In [ ]:
# Pakete installieren + GPU pruefen
!pip -q install duckdb transformers adapters

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

## 5 - Autoren und Paper laden

Alle produktiven Autoren auswaehlen und ihre Paper aus der Datenbank ziehen.

In [ ]:
# Produktive Autoren finden
import duckdb, numpy as np

TBL         = "openalex_ai_raw_v1_0"   # Tabelle mit der reichen Struktur
MIN_PAPERS  = 3                         # Autor muss >= so viele Paper haben
N_AUTHORS   = 150                       # so viele Autoren zufaellig waehlen
SEED        = 42

con = duckdb.connect(LOCAL_DB, read_only=True)

# Autor -> Paper entpacken, nur Paper mit Titel + Abstract
con.execute(f"""
    CREATE OR REPLACE TEMP VIEW author_paper AS
    WITH exploded AS (
        SELECT id AS work_id,
               unnest(authorships) AS a
        FROM {TBL}
        WHERE title IS NOT NULL AND abstract_inverted_index IS NOT NULL
    )
    SELECT a.author.id AS author_id, a.author.display_name AS author_name, work_id
    FROM exploded
    WHERE a.author.id IS NOT NULL
""")

# Produktive Autoren finden
prod = con.execute(f"""
    SELECT author_id, author_name, COUNT(DISTINCT work_id) AS n_paper
    FROM author_paper
    GROUP BY author_id, author_name
    HAVING COUNT(DISTINCT work_id) >= {MIN_PAPERS}
    ORDER BY n_paper DESC
""").fetchall()

print(f"{len(prod)} Autoren mit >= {MIN_PAPERS} Papern.")

rng = np.random.default_rng(SEED)

# VOLLER LAUF: alle produktiven Autoren (>= MIN_PAPERS), keine Zufallsstichprobe.
chosen_ids = [p[0] for p in prod]
print(f"{len(chosen_ids)} Autoren (VOLLER LAUF, alle mit >= {MIN_PAPERS} Papern).")

In [ ]:
# Paper der Autoren laden (mit IDs und Datum)
ph = ",".join(["?"] * len(chosen_ids))
rows = con.execute(f"""
    WITH work_ids AS (
        SELECT DISTINCT work_id FROM author_paper WHERE author_id IN ({ph})
    )
    SELECT
        w.id                                          AS work_id,
        w.title,
        w.abstract_inverted_index                     AS abs_idx,
        w.publication_year                            AS year,
        COALESCE(strftime(w.publication_date, '%Y-%m-%d'),
                 CAST(w.publication_year AS VARCHAR) || '-01-01') AS pub_date,
        w.primary_location.source.id                  AS journal_id,
        w.primary_location.source.display_name        AS journal_name,
        w.primary_location.source.host_organization   AS publisher_id,
        w.primary_location.source.is_in_doaj          AS in_doaj
    FROM {TBL} w
    JOIN work_ids ON work_ids.work_id = w.id
""", chosen_ids).fetchall()

print(f"{len(rows)} Paper der gewaehlten Autoren.")

## 6 - Abstract wiederherstellen

OpenAlex speichert Abstracts als Wort-Positionen. Wir setzen daraus wieder normalen Text zusammen.

In [ ]:
# Abstract aus Wort-Positionen zusammensetzen
def reconstruct_abstract(inv):
    if not inv:
        return ""
    pairs = []
    for word, positions in inv.items():
        for p in positions:
            pairs.append((p, word))
    pairs.sort(key=lambda x: x[0])
    return " ".join(w for _, w in pairs)

titles, abstracts, meta = [], [], []
for work_id, title, abs_idx, year, pub_date, journal_id, journal_name, publisher_id, in_doaj in rows:
    abstract = reconstruct_abstract(abs_idx)
    if not abstract.strip():
        continue
    titles.append(title)
    abstracts.append(abstract)
    meta.append({"work_id": work_id, "year": year, "date": pub_date,
                 "journal_id": journal_id, "journal_name": journal_name,
                 "publisher_id": publisher_id, "in_doaj": in_doaj})

print(f"{len(titles)} Paper mit nutzbarem Abstract.")
print("\nBeispiel-Titel   :", titles[0][:90])
print("Beispiel-Abstract:", abstracts[0][:200], "...")

## 7 - Paper einbetten

SPECTER2 laedt und macht aus jedem Paper einen 768-Zahlen-Vektor. Auf GPU ein paar Minuten.

In [ ]:
# SPECTER2 laden und alle Paper einbetten
from transformers import AutoTokenizer
from adapters import AutoAdapterModel

tok   = AutoTokenizer.from_pretrained("allenai/specter2_base")
model = AutoAdapterModel.from_pretrained("allenai/specter2_base")
model.load_adapter("allenai/specter2", source="hf", set_active=True)
model.to(DEVICE).eval()

def embed(titles, abstracts, batch_size=32):
    texts = [(t or "") + tok.sep_token + (a or "") for t, a in zip(titles, abstracts)]
    out = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inp = tok(batch, padding=True, truncation=True,
                  return_tensors="pt", max_length=512).to(DEVICE)
        with torch.no_grad():
            o = model(**inp)
        out.append(o.last_hidden_state[:, 0, :].cpu().numpy())  # CLS
        print(f"  {min(i+batch_size, len(texts))}/{len(texts)}", end="\r")
    return np.vstack(out)

X = embed(titles, abstracts)
X = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)   # L2-normalisieren
print(f"\nEmbeddings: {X.shape}   (Paper x 768)")

## 8 - Aehnlichkeit (Kernel)

Der RBF-Kernel misst die Aehnlichkeit zwischen zwei Vektoren. Wir rechnen nur die noetigen Vergleiche, nie die riesige Gesamtmatrix - das spart Speicher.

In [ ]:
# Kernel-Aehnlichkeit, speicherschonend
_rng = np.random.default_rng(0)
_m = min(3000, len(X))
_idx = _rng.choice(len(X), _m, replace=False)
_Xs = X[_idx]
_s = np.sum(_Xs**2, axis=1)
_D2 = np.maximum(_s[:, None] + _s[None, :] - 2 * _Xs @ _Xs.T, 0.0)
sig2 = float(np.median(_D2[np.triu_indices(_m, k=1)]))
print(f"sigma^2 (Median-Heuristik, Stichprobe {_m}) = {sig2:.4f}")

def kernel_group_mean(idxs):
    """Mittlere paarweise RBF-Aehnlichkeit innerhalb einer kleinen Gruppe."""
    if len(idxs) < 2:
        return None
    sub = X[idxs]
    s = np.sum(sub**2, axis=1)
    D2 = np.maximum(s[:, None] + s[None, :] - 2 * sub @ sub.T, 0.0)
    K = np.exp(-0.5 * D2 / sig2)
    iu = np.triu_indices(len(idxs), k=1)
    return float(np.mean(K[iu]))

def sim_to_all(i):
    """RBF-Aehnlichkeit eines Ankers zu ALLEN Papern (ein Vektor, keine Matrix)."""
    d = X - X[i]
    D2 = np.sum(d**2, axis=1)
    return np.exp(-0.5 * D2 / sig2)

# Zentrale Zuordnungen (von hier an gebraucht)
row_of        = {m["work_id"]: i for i, m in enumerate(meta)}
journal_id_of = {m["work_id"]: m["journal_id"] for m in meta}
emb_ids       = set(row_of)

## 9 - Kurzer Test + erste Q1-Zahl

Pruefen, ob aehnliche Paper wirklich thematisch passen, und die mittlere Aehnlichkeit von Papern desselben Autors im selben Journal messen.

In [ ]:
# Aehnlichste Paper zu einem Anker
def most_similar(i, k=3):
    sims = sim_to_all(i); sims[i] = -1
    for j in np.argsort(sims)[::-1][:k]:
        print(f"   {sims[j]:.3f}  {titles[j][:75]}")

for a in [0, 1, 2]:
    print(f"\nAnker: {titles[a][:75]}")
    most_similar(a)

In [ ]:
# Q1: mittlere Themen-Aehnlichkeit im selben Journal
from collections import defaultdict

author_works = defaultdict(list)
for aid in chosen_ids:
    for (w,) in con.execute(
        "SELECT DISTINCT work_id FROM author_paper WHERE author_id = ?", [aid]
    ).fetchall():
        if w in emb_ids:
            author_works[aid].append(w)

def author_intra_journal(work_ids):
    by_j = defaultdict(list)
    for w in work_ids:
        by_j[journal_id_of[w]].append(row_of[w])
    vals = [kernel_group_mean(idxs) for idxs in by_j.values() if len(idxs) >= 2]
    vals = [v for v in vals if v is not None]
    return float(np.mean(vals)) if vals else None

scores = [(a, author_intra_journal(ws)) for a, ws in author_works.items()]
scores = [(a, s) for a, s in scores if s is not None]
if scores:
    arr = np.array([s for _, s in scores])
    print(f"Autoren mit >=2 Papern im selben Journal: {len(scores)}")
    print(f"Mittlere Intra-Journal-Aehnlichkeit: {arr.mean():.3f} (SD {arr.std():.3f})")
else:
    print("Keine Autoren mit >=2 Papern im selben Journal.")

## 10 - Was bedeuten die Zahlen?

topic_match liegt zwischen 0 und 1: hoch = thematisch aehnlich, niedrig = fremd. Weil der Datensatz nur KI-Paper enthaelt, sind die Werte generell hoch - wir lesen **Unterschiede**, nicht absolute Hoehen.

## 11 - Anschluss an die anderen Layer

Alle drei Layer verbinden sich ueber die **work_id**. Unser topic_match ist die Themen-Variable, die der Probabilistic-Layer fuer die Q3-Frage nutzt.

## 12 - Schluessel-Tabelle

Alle Autor-Paper-Zeilen mit IDs - die Basis, die der Logic-Layer braucht.

In [ ]:
# Schluessel-Tabelle (alle Autor-Paper-Zeilen)
keys = con.execute(f"""
    WITH exploded AS (
        SELECT id AS work_id,
               publication_year                            AS year,
               primary_location.source.id                  AS journal_id,
               primary_location.source.display_name        AS journal_name,
               primary_location.source.host_organization   AS publisher_id,
               primary_location.source.is_in_doaj          AS in_doaj,
               unnest(authorships)                          AS a
        FROM {TBL}
    )
    SELECT work_id, year, journal_id, journal_name, publisher_id, in_doaj,
           a.author.id           AS author_id,
           a.author.display_name AS author_name
    FROM exploded
    WHERE a.author.id IS NOT NULL
""").fetchall()

print(f"{len(keys)} Autor-Paper-Zeilen.")
print("Spalten: work_id, year, journal_id, journal_name, publisher_id, in_doaj, author_id, author_name")
print("Beispiel:", keys[0])

## 13 - Ergebnisse exportieren

Ergebnisse als kleine CSVs nach Drive (nicht zurueck in die Datenbank - die bleibt ein Wegwerf-Artefakt).

In [ ]:
# Export: Q1-Ergebnis als CSV
import csv, shutil

rows_out = []
for aid, ws in author_works.items():
    by_j = defaultdict(list)
    for w in ws:
        by_j[journal_id_of[w]].append(w)
    for jid, wids in by_j.items():
        if len(wids) >= 2:
            val = kernel_group_mean([row_of[w] for w in wids])
            rows_out.append({
                "author_id": aid,
                "journal_id": jid,
                "n_papers": len(wids),
                "topic_match_intra": round(val, 4),
                "work_ids": ";".join(wids),
            })

LOCAL_CSV = "/content/results_q1_topic_match.csv"
with open(LOCAL_CSV, "w", newline="") as f:
    wr = csv.DictWriter(f, fieldnames=["author_id","journal_id","n_papers",
                                       "topic_match_intra","work_ids"])
    wr.writeheader(); wr.writerows(rows_out)

shutil.copy(LOCAL_CSV, "/content/drive/MyDrive/Academic-Journals/results_q1_topic_match.csv")
print(f"{len(rows_out)} Autor-Journal-Zeilen exportiert (voller Lauf).")
if rows_out:
    print("Beispiel:", rows_out[0])

In [ ]:
# Export: Schluessel-Tabelle als CSV
import csv, shutil

LOCAL_KEYS = "/content/keys_author_paper.csv"
with open(LOCAL_KEYS, "w", newline="") as f:
    wr = csv.writer(f)
    wr.writerow(["work_id","year","journal_id","journal_name",
                 "publisher_id","in_doaj","author_id","author_name"])
    wr.writerows(keys)

shutil.copy(LOCAL_KEYS, "/content/drive/MyDrive/Academic-Journals/keys_author_paper.csv")
print(f"{len(keys)} Zeilen exportiert nach keys_author_paper.csv")

## 14 - Zeitkorrektes topic_match

topic_match nur aus Papern VOR dem jeweiligen Zeitpunkt - so nutzt kein Wert Information aus der Zukunft (kein Leakage).

In [ ]:
# Zeitkorrektes topic_match (nur vergangene Paper)
import bisect
from collections import defaultdict

date_of = {m["work_id"]: m["date"] for m in meta}

def cumulative_centroids(items):
    """items: list of (date, row). -> centroid_before(t): Zentroid der
    Embeddings mit date STRIKT < t, sonst None."""
    items = sorted(items)
    dates = [d for d, _ in items]
    rows  = [r for _, r in items]
    cs = np.cumsum(X[rows], axis=0) if rows else None
    def centroid_before(t):
        k = bisect.bisect_left(dates, t)       # Anzahl Paper mit date < t
        return None if k == 0 else cs[k-1] / k
    return centroid_before

# Profile pro Journal (aus allen embeddeten Papern) und pro Autor
journal_papers = defaultdict(list)
for i, m in enumerate(meta):
    journal_papers[m["journal_id"]].append((m["date"], i))

author_papers = defaultdict(list)
for aid, ws in author_works.items():
    for w in ws:
        author_papers[aid].append((date_of[w], row_of[w]))

j_centroid = {j: cumulative_centroids(v) for j, v in journal_papers.items()}
a_centroid = {a: cumulative_centroids(v) for a, v in author_papers.items()}

def topic_match_temporal(aid, jid, t):
    ac = a_centroid[aid](t) if aid in a_centroid else None
    jc = j_centroid[jid](t) if jid in j_centroid else None
    if ac is None or jc is None:
        return None
    d2 = float(np.sum((ac - jc) ** 2))
    return float(np.exp(-0.5 * d2 / sig2))

# Gelegenheiten = jedes Autor-Paper
occasions = []
for aid, ws in author_works.items():
    for w in ws:
        occasions.append((aid, journal_id_of[w], date_of[w], w))

vals = [topic_match_temporal(a, j, t) for a, j, t, _ in occasions]
n_def = sum(v is not None for v in vals)
print(f"{len(occasions)} Gelegenheiten, davon {n_def} mit Vergangenheits-Profil.")
if n_def:
    arr = np.array([v for v in vals if v is not None])
    print(f"topic_match(t): Mittel {arr.mean():.3f}, SD {arr.std():.3f}")

In [ ]:
# Export: zeitkorrekte Event-Tabelle
import csv, shutil

LOCAL = "/content/results_topic_match_temporal.csv"
with open(LOCAL, "w", newline="") as f:
    wr = csv.writer(f)
    wr.writerow(["author_id","journal_id","t","work_id","topic_match_t"])
    for (aid, jid, t, w), v in zip(occasions, vals):
        wr.writerow([aid, jid, t, w, "" if v is None else round(v, 4)])

shutil.copy(LOCAL, "/content/drive/MyDrive/Academic-Journals/results_topic_match_temporal.csv")
print(f"{len(occasions)} Zeilen exportiert (zeitkorrekt, fuer die Event-Tabelle).")